# Перенос датасета Luzitania в MERA

Ноутбук переносит публичный математический бенчмарк **Luzitania** из приватного HF-репозитория в общий репозиторий MERA, а параллельно — собирает локальные `test.json` и `shots.json` в формате MERA.

**Источник:** `a141592/Luzitania` (private)  
**Назначение:** `MERA-evaluation/Luzitania` (public)

## Порядок шагов

1. Скачивание датасета из приватного HF-репозитория.
2. **Сразу после скачивания — перезаливка исходных данных в `MERA-evaluation/Luzitania`** (без изменений, как есть). Это первый шаг переноса.
3. Инспекция структуры — сплиты, колонки, примеры.
4. Конвертация в формат MERA: каждый пример → `{instruction, inputs:{question}, outputs, meta:{id, source, spec}}`.
5. Опциональная замена строкового `instruction` на его порядковый номер в каноническом списке промптов.
6. Sanity-проверки (размер, структура, ID, баланс промптов, source-значения).
7. Сохранение локально в `./data/test.json` и `./data/shots.json`.

Ноутбук запускается **сверху вниз**.

## 1. Установка зависимостей и импорты

In [ ]:
# Раскомментируйте в Colab или при первом запуске:
# !pip install -q datasets huggingface_hub

In [24]:
import json
import os
import sys
from collections import Counter
from pathlib import Path
from typing import Any

def info(msg: str) -> None:
    print(f"[INFO]  {msg}")

def warn(msg: str) -> None:
    print(f"[WARN]  {msg}")

def fail(msg: str) -> None:
    print(f"\033[91m[ERROR]\033[0m {msg}")

## 2. Конфигурация: репозитории, токены, пути

Токены лучше задавать через переменные окружения `HF_TOKEN_READ` и `HF_TOKEN_WRITE`. 

Флаги:
- `REPLACE_INSTRUCTION_WITH_INDEX` — заменять строковый `instruction` на индекс из `PROMPTS` (правильное поведение по спецификации MERA).
- `DO_UPLOAD` — пушить ли исходный датасет в `MERA-evaluation/Luzitania`.

In [25]:
SRC_REPO = "a141592/Luzitania"           # источник (private)
DST_REPO = "MERA-evaluation/Luzitania"   # назначение (public)

TOKEN_READ = os.environ.get("HF_TOKEN_READ") or "hf_"          # подставьте read-токен
TOKEN_WRITE = os.environ.get("HF_TOKEN_WRITE") or "hf_"   # подставьте write-токен

DATA_DIR = Path("./data")
TEST_OUT = DATA_DIR / "test.json"
SHOTS_OUT = DATA_DIR / "shots.json"

REPLACE_INSTRUCTION_WITH_INDEX = True   # True — int-индекс; False — оставить строку
DO_UPLOAD = True                         # True — пушим исходный сет на HF; False — пропускаем

ACCESS = "public"
EXPECTED_SPLITS = ("test", "shots")
EXPECTED_SOURCES = {
    "MathArena",
    "turgor",
    "olympiads",
    "olympic_reason",
    "chinese_olympiads_2002_2006",
    "example",
}

print(f"SRC: {SRC_REPO}")
print(f"DST: {DST_REPO}")
print(f"Replace instruction with index: {REPLACE_INSTRUCTION_WITH_INDEX}")
print(f"Upload to HF: {DO_UPLOAD}")
print(f"Local output: {TEST_OUT}, {SHOTS_OUT}")

SRC: a141592/Luzitania
DST: MERA-evaluation/Luzitania
Replace instruction with index: True
Upload to HF: True
Local output: data/test.json, data/shots.json


## 3. Скачивание датасета из приватного HF

Используем `datasets.load_dataset` с токеном чтения для доступа к приватному репо.

In [26]:
from datasets import load_dataset

info(f"Загрузка датасета из {SRC_REPO} ...")
ds = load_dataset(SRC_REPO, token=TOKEN_READ)
info(f"Загружены сплиты: {list(ds.keys())}")
ds

[INFO]  Загрузка датасета из a141592/Luzitania ...
[INFO]  Загружены сплиты: ['shots', 'test']


DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 258
    })
})

## 4. Перезаливка исходного датасета в MERA-evaluation/Luzitania

Первый шаг переноса — публикация исходного датасета в общий репозиторий MERA **как есть, без каких-либо преобразований**.

Что делает ячейка:

1. **Проверяет существование** репозитория `MERA-evaluation/Luzitania`. Организация `MERA-evaluation` существует, а вот конкретный датасет в ней — не факт; нужно учесть оба случая.
2. **Создаёт репозиторий**, если его нет:
   - `repo_type="dataset"`
   - `private=True` — пока приватный
   - `exist_ok=True` — не падать, если кем-то уже создан
3. **Пушит исходный `ds`** через `push_to_hub` без изменений. На HF Hub данные сохраняются как parquet-сплиты — стандартный формат для датасетов на Hub.

Дальше в ноутбуке мы продолжим работать с локальным `ds` для подготовки `test.json`/`shots.json`, а сам датасет уже будет лежать в MERA-репозитории.

In [5]:
DST_REPO

'MERA-evaluation/Luzitania'

In [11]:

if not DO_UPLOAD:
    info("DO_UPLOAD = False, пропускаем заливку исходного датасета.")
else:
    from huggingface_hub import HfApi, create_repo
    from huggingface_hub.utils import RepositoryNotFoundError

    api = HfApi(token=TOKEN_WRITE)

    # 1) Проверяем, существует ли уже репо
    info(f"Проверка существования {DST_REPO} ...")
    try:
        api.dataset_info(DST_REPO)
        info(f"Репозиторий {DST_REPO} уже существует — будем пушить туда.")
    except RepositoryNotFoundError:
        info(f"Репозиторий {DST_REPO} не найден, создаём ...")
        create_repo(
            repo_id=DST_REPO,
            repo_type="dataset",
            private=True,
            exist_ok=True,
            token=TOKEN_WRITE,
        )
        info(f"Репозиторий {DST_REPO} создан.")
    except Exception as e:
        fail(f"Не удалось проверить статус репозитория: {e}")
        raise

    # 2) Пушим исходный датасет КАК ЕСТЬ
    info(f"Заливка исходного датасета в {DST_REPO} ...")
    ds.push_to_hub(DST_REPO, token=TOKEN_WRITE, private=False)
    info("Заливка исходного датасета завершена.")
    print(f"Проверьте результат: https://huggingface.co/datasets/{DST_REPO}")

[INFO]  Проверка существования MERA-evaluation/Luzitania ...
[INFO]  Репозиторий MERA-evaluation/Luzitania уже существует — будем пушить туда.
[INFO]  Заливка исходного датасета в MERA-evaluation/Luzitania ...


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1313.59ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 574.80ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


[INFO]  Заливка исходного датасета завершена.
Проверьте результат: https://huggingface.co/datasets/MERA-evaluation/Luzitania


## 5. Инспекция: сплиты, колонки, примеры

Смотрим на структуру данных перед конвертацией: какие сплиты есть, какие у них колонки и как выглядят первые примеры.

In [27]:
def inspect_dataset(ds, n_samples: int = 3) -> None:
    """Печатает структуру загруженного датасета: сплиты, колонки, примеры."""
    print("=" * 70)
    print(" " * 25 + "ИНСПЕКЦИЯ ДАТАСЕТА")
    print("=" * 70)

    available_splits = set(ds.keys())
    missing = set(EXPECTED_SPLITS) - available_splits
    extra = available_splits - set(EXPECTED_SPLITS)

    if missing:
        warn(f"Отсутствуют ожидаемые сплиты: {sorted(missing)}.")
    if extra:
        warn(f"Найдены дополнительные сплиты: {sorted(extra)}. Они будут проигнорированы.")
    if not missing:
        info(f"Сплиты {EXPECTED_SPLITS} присутствуют.")

    for split_name in EXPECTED_SPLITS:
        if split_name not in ds:
            continue
        split = ds[split_name]
        print(f"\n--- Сплит '{split_name}' ---")
        print(f"  Размер: {len(split)} примеров")
        print(f"  Колонки: {split.column_names}")

        if len(split) > 0:
            print(f"  Первые {min(n_samples, len(split))} примера:")
            for i in range(min(n_samples, len(split))):
                ex = split[i]
                ex_short = {
                    k: (v if not isinstance(v, str) or len(v) <= 120
                        else v[:120] + "...")
                    for k, v in ex.items()
                }
                print(f"    [{i}] {json.dumps(ex_short, ensure_ascii=False)}")

inspect_dataset(ds, n_samples=3)

                         ИНСПЕКЦИЯ ДАТАСЕТА
[INFO]  Сплиты ('test', 'shots') присутствуют.

--- Сплит 'test' ---
  Размер: 258 примеров
  Колонки: ['instruction', 'inputs', 'outputs', 'meta']
  Первые 3 примера:
    [0] {"instruction": "Решите следующую задачу. Ответ запишите в виде целого числа, если в задаче не указано иное. {question}", "inputs": {"question": "Пусть \\(n\\) — натуральное число, а \\(p\\) — простое. Через \\(n\\) и \\(p\\) обозначим \\(k(n,p)\\) как наибольшее неотрицательное целое число \\(k\\), для которого существует многочлен \\(P(x)\\) с целыми коэффициентами, удовлетворяющий условиям:  \n- Коэффициент при \\(x^n\\) в \\(P(x)\\) равен \\(1\\).  \n- \\(p^k\\) делит \\(P(x)\\) для всех целых \\(x\\).\n\nВычислите\n\\[\n\\sum_{n=11}^{15} \\sum_{p \\in \\{11,13\\}} k(n,p).\n\\]\nкак целое число."}, "outputs": "138", "meta": {"id": 1, "source": "apex", "spec": null}}
    [1] {"instruction": "Найдите решение следующей задачи: {question}. Запишите ответ целым числом (е

## 6. Канонический список промптов

10 промптов из спецификации Luzitania. Порядок зафиксирован — индекс в этом списке используется как `instruction: int` в формате MERA.

Если в исходных HF-данных строка промпта не совпадёт байт-в-байт ни с одним из этих — будет напечатано предупреждение, а `instruction` останется строкой (страховка от молчаливого сбоя).

In [28]:
PROMPTS: list[str] = [
    "Решите следующую задачу. Ответ запишите в виде целого числа, если в задаче не указано иное. {question}",
    "Найдите решение следующей задачи: {question}. Запишите ответ целым числом (если в условии задачи не указано иное).",
    "Дано условие задачи: {question}. Найдите решение. Ответ представьте в виде целого числа. (Исключения оговорены в условии.)",
    "Дайте численный ответ. По умолчанию ответ — целое число. {question}",
    "Решите предложенную задачу. Ответом служит целое число, если в условии задачи нет дополнительных указаний. {question}",
    "Реши задачу по математике: {question} \nВ ответе выведи только целое число. Если в задаче указан другой формат ответа, следуй указаниям задачи.",
    "Найди решение задачи: {question}. Если не сказано иного, ответ запиши целым числом, без дополнительных символов и пояснений.",
    "Ответ к следующей задаче необходимо вывести целым числом, без специальных чисел и пояснений. Условие задачи: {question}",
    "Прочитайте задачу: {question}. Найдите ответ. По умолчанию ответ — целое число. В случае нестандартного формата ответа следуйте указаниям в условии.",
    "Прочти задачу и найди решение. {question}. \n Результат запиши целым числом, если в условии не требуется иной формат ответа.",
]

PROMPT_TO_INDEX: dict[str, int] = {p: i for i, p in enumerate(PROMPTS)}

print(f"Загружено {len(PROMPTS)} канонических промптов.")

Загружено 10 канонических промптов.


## 7. Конвертация в формат MERA

Каждый пример приводим к формату:
```json
{
  "instruction": <int | str>,
  "inputs": {"question": "..."},
  "outputs": "...",
  "meta": {"id": int, "source": "...", "spec": ""}
}
```

Функция `normalize_example` поддерживает два варианта структуры HF-датасета:
- **Вложенный** — поля `inputs` и `meta` уже есть как объекты.
- **Плоский** — поля лежат на верхнем уровне (`question`, `id`, `source`, `spec`).

Если `inputs`/`meta` хранятся как JSON-строки (бывает при сохранении в parquet), они автоматически парсятся.

Если `REPLACE_INSTRUCTION_WITH_INDEX=True`, строковый `instruction` заменяется на индекс в `PROMPTS`. Если промпт не найден — печатается предупреждение, а строка остаётся.

In [39]:
def normalize_example(example: dict[str, Any], replace_instruction_with_index: bool) -> dict[str, Any]:
    """Приводит один HF-пример к формату MERA."""
    instruction = example.get("instruction")
    inputs = example.get("inputs")
    outputs = example.get("outputs")
    meta = example.get("meta")

    # Если inputs/meta — JSON-строки, разбираем
    if isinstance(inputs, str):
        inputs = json.loads(inputs)
    if isinstance(meta, str):
        meta = json.loads(meta)

    # Поддержка плоского формата HF: поднимаем question/id/source/spec в inputs/meta
    if inputs is None and "question" in example:
        inputs = {"question": example["question"]}

    if meta is None:
        meta = {
            k: example.get(k, "")
            for k in ("id", "source", "spec")
        }

    # Нормализуем meta: None в source/spec заменяем на ""
    if isinstance(meta, dict):
        meta = {
            k: "" if (k in ("source", "spec") and meta.get(k) is None) else meta.get(k, "")
            for k in ("id", "source", "spec")
        }

    # Замена instruction на индекс (опционально)
    if replace_instruction_with_index and isinstance(instruction, str):
        idx = PROMPT_TO_INDEX.get(instruction)
        if idx is None:
            warn(
                f"Промпт не найден в каноническом списке "
                f"(meta.id={meta.get('id') if isinstance(meta, dict) else '?'}). "
                "Оставлен строковый вариант."
            )
        else:
            instruction = idx

    return {
        "instruction": instruction,
        "inputs": inputs,
        "outputs": outputs,
        "meta": meta,
    }


def split_to_mera_format(split, replace_instruction_with_index: bool) -> list[dict[str, Any]]:
    return [normalize_example(ex, replace_instruction_with_index) for ex in split]


if "test" not in ds or "shots" not in ds:
    raise RuntimeError(
        f"В исходном датасете отсутствует один из ожидаемых сплитов: {EXPECTED_SPLITS}. "
        f"Найдено: {list(ds.keys())}."
    )

info("Конвертация test ...")
test_data = split_to_mera_format(ds["test"], REPLACE_INSTRUCTION_WITH_INDEX)
info("Конвертация shots ...")
shots_data = split_to_mera_format(ds["shots"], REPLACE_INSTRUCTION_WITH_INDEX)
info(f"Готово: test={len(test_data)}, shots={len(shots_data)}")

print("\n--- Пример из test ---")
print(json.dumps(test_data[0], indent=2, ensure_ascii=False)[:500])
print("\n--- Пример из shots ---")
print(json.dumps(shots_data[0], indent=2, ensure_ascii=False)[:500])

[INFO]  Конвертация test ...
[INFO]  Конвертация shots ...
[INFO]  Готово: test=258, shots=8

--- Пример из test ---
{
  "instruction": 0,
  "inputs": {
    "question": "Пусть \\(n\\) — натуральное число, а \\(p\\) — простое. Через \\(n\\) и \\(p\\) обозначим \\(k(n,p)\\) как наибольшее неотрицательное целое число \\(k\\), для которого существует многочлен \\(P(x)\\) с целыми коэффициентами, удовлетворяющий условиям:  \n- Коэффициент при \\(x^n\\) в \\(P(x)\\) равен \\(1\\).  \n- \\(p^k\\) делит \\(P(x)\\) для всех целых \\(x\\).\n\nВычислите\n\\[\n\\sum_{n=11}^{15} \\sum_{p \\in \\{11,13\\}} k(n,p).\n\\]\nкак

--- Пример из shots ---
{
  "instruction": 0,
  "inputs": {
    "question": "Пусть $(A,\\Theta)$ будет очень общим главным поляризованным комплексным абелевым многообразием размерности $6$. Положим $\\theta:=c_1(\\Theta)\\in H^2(A,\\mathbb Z)$ и определим класс когомологий минимальной кривой как\n\\[\n\\gamma:=\\frac{\\theta^5}{5!}\\in H^{10}(A,\\mathbb Z),\n\\]\nкоторый являетс

## 8. Sanity-проверки готовых данных

Стандартные проверки бенчмарка MERA (по урокам ревью SAGE):

1. **Размер test** — минимум 300.
2. **Структура примеров** — у каждого ровно 4 ключа: `instruction`, `inputs`, `outputs`, `meta`.
3. **inputs.question** — обязательный ключ в `inputs`.
4. **meta.id** — int, сквозная нумерация с 1, без дубликатов (объединённо для test + shots).
5. **instruction в каноническом списке** — либо int в `[0, len(PROMPTS))`, либо строка из `PROMPTS`.
6. **Баланс instruction** — отклонение от равномерного распределения не больше 20%.
7. **outputs не пустой**.
8. **meta.source** — значения из ожидаемого списка.

In [40]:
def sanity_checks(test_data: list[dict[str, Any]], shots_data: list[dict[str, Any]]) -> None:
    print("=" * 70)
    print(" " * 25 + "SANITY CHECKS")
    print("=" * 70)

    results: list[tuple[str, str, str]] = []

    def add(name: str, ok: bool, msg: str = "", warn_only: bool = False) -> None:
        results.append((name, "PASS" if ok else ("WARN" if warn_only else "FAIL"), msg))

    n_test = len(test_data)
    n_shots = len(shots_data)

    # 1. Размер test
    add(
        "01. Размер test",
        n_test >= 300,
        f"test содержит {n_test} примеров (требование MERA: минимум 300).",
        warn_only=(n_test >= 100),
    )

    # 2. Структура примеров
    required_keys = {"instruction", "inputs", "outputs", "meta"}
    bad = [
        i for i, ex in enumerate(test_data + shots_data)
        if not isinstance(ex, dict) or set(ex.keys()) != required_keys
    ]
    add(
        "02. Структура примеров",
        not bad,
        f"Некорректные примеры (индексы): {bad[:5]}" if bad else "Все примеры имеют корректный набор ключей.",
    )

    # 3. inputs.question
    bad_inputs = []
    for i, ex in enumerate(test_data + shots_data):
        inputs = ex.get("inputs") if isinstance(ex, dict) else None
        if not isinstance(inputs, dict) or "question" not in inputs:
            bad_inputs.append(i)
    add(
        "03. inputs.question",
        not bad_inputs,
        f"Нет inputs.question в примерах: {bad_inputs[:5]}" if bad_inputs
        else "inputs.question есть во всех примерах.",
    )

    # 4. meta.id
    all_ids = []
    for ex in test_data + shots_data:
        meta = ex.get("meta") if isinstance(ex, dict) else None
        if isinstance(meta, dict) and isinstance(meta.get("id"), int):
            all_ids.append(meta["id"])
    ids_ok = bool(all_ids) and (
        len(all_ids) == n_test + n_shots
        and min(all_ids) == 1
        and max(all_ids) == n_test + n_shots
        and len(set(all_ids)) == len(all_ids)
    )
    add(
        "04. meta.id (сквозная нумерация с 1)",
        ids_ok,
        f"Найдено {len(all_ids)} id, диапазон "
        f"{min(all_ids) if all_ids else '?'}..{max(all_ids) if all_ids else '?'}, "
        f"уникальных: {len(set(all_ids))}",
    )

    # 5. instruction в каноническом списке
    bad_instr = []
    for i, ex in enumerate(test_data + shots_data):
        instr = ex.get("instruction")
        if isinstance(instr, int):
            if not (0 <= instr < len(PROMPTS)):
                bad_instr.append(i)
        elif isinstance(instr, str):
            if instr not in PROMPT_TO_INDEX:
                bad_instr.append(i)
        else:
            bad_instr.append(i)
    add(
        "05. instruction в каноническом списке",
        not bad_instr,
        f"Несоответствующие instruction в примерах: {bad_instr[:5]}" if bad_instr
        else "Все instruction соответствуют PROMPTS.",
    )

    # 6. Баланс instruction
    instr_counter: Counter = Counter()
    for ex in test_data:
        instr = ex.get("instruction")
        instr_key = instr if isinstance(instr, int) else PROMPT_TO_INDEX.get(instr, -1)
        instr_counter[instr_key] += 1
    if instr_counter:
        expected = n_test / len(instr_counter)
        max_dev = max(abs(c - expected) / expected for c in instr_counter.values()) * 100
        add(
            "06. Баланс instruction в test",
            max_dev <= 20,
            f"Используется {len(instr_counter)} промптов, max отклонение {max_dev:.1f}%.",
            warn_only=True,
        )

    # 7. outputs не пустой
    empty_outputs = sum(
        1 for ex in test_data + shots_data
        if not isinstance(ex.get("outputs"), str) or not ex["outputs"].strip()
    )
    add(
        "07. outputs не пустой",
        empty_outputs == 0,
        f"Найдено пустых outputs: {empty_outputs}",
    )

    # 8. meta.source значения
    bad_sources = []
    found_sources: Counter = Counter()
    for i, ex in enumerate(test_data + shots_data):
        meta = ex.get("meta") if isinstance(ex, dict) else None
        if isinstance(meta, dict):
            src = meta.get("source")
            found_sources[src] += 1
            if src not in EXPECTED_SOURCES:
                bad_sources.append((i, src))
    add(
        "08. meta.source значения",
        not bad_sources,
        f"Найдены неожиданные значения source: {bad_sources[:5]}" if bad_sources
        else f"Распределение: {dict(found_sources)}",
        warn_only=True,
    )

    for name, status, msg in results:
        label = {
            "PASS": "\033[92m[PASS]\033[0m",
            "WARN": "\033[93m[WARN]\033[0m",
            "FAIL": "\033[91m[FAIL]\033[0m",
        }[status]
        print(f"  {label} {name}")
        if msg:
            print(f"          {msg}")

    n_fail = sum(1 for _, s, _ in results if s == "FAIL")
    n_warn = sum(1 for _, s, _ in results if s == "WARN")
    print()
    if n_fail:
        print(f"\033[91mИТОГ: {n_fail} критических замечаний\033[0m, исправьте перед заливкой.")
    elif n_warn:
        print(f"\033[93mИТОГ: {n_warn} предупреждений\033[0m — проверьте детали.")
    else:
        print("\033[92mИТОГ: все проверки пройдены\033[0m.")


sanity_checks(test_data, shots_data)

                         SANITY CHECKS
  [WARN] 01. Размер test
          test содержит 258 примеров (требование MERA: минимум 300).
  [PASS] 02. Структура примеров
          Все примеры имеют корректный набор ключей.
  [PASS] 03. inputs.question
          inputs.question есть во всех примерах.
  [PASS] 04. meta.id (сквозная нумерация с 1)
          Найдено 266 id, диапазон 1..266, уникальных: 266
  [PASS] 05. instruction в каноническом списке
          Все instruction соответствуют PROMPTS.
  [PASS] 06. Баланс instruction в test
          Используется 10 промптов, max отклонение 18.6%.
  [PASS] 07. outputs не пустой
          Найдено пустых outputs: 0
  [WARN] 08. meta.source значения
          Найдены неожиданные значения source: [(0, 'apex'), (1, 'apex'), (2, 'apex'), (3, 'apex'), (4, 'apex')]

ИТОГ: 2 предупреждений — проверьте детали.


## 9. Запись локальных файлов test.json и shots.json

Сохраняем в формате MERA:
```json
    Для test:  {"access": access, "data": data}
    Для shots: {"data": data}
```

Параметры записи:
- `indent=4` — по спецификации MERA.
- `ensure_ascii=False` — кириллица сохраняется как есть, а не в виде `\uXXXX`.
- Перевод строки в конце файла (POSIX-стандарт).

Файлы попадут в `./data/test.json` и `./data/shots.json`.

In [43]:
def write_json(
    path: Path,
    data: list[dict[str, Any]],
    access: str = "public",
    include_access: bool = True,
) -> None:
    """Записывает данные в MERA-формат.

    Для test:  {"access": access, "data": data}
    Для shots: {"data": data}
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = {"data": data}
    if include_access:
        payload["access"] = access

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=4)
        f.write("\n")

    info(f"Записан файл {path} ({len(data)} примеров).")


write_json(TEST_OUT, test_data, access=ACCESS, include_access=True)
write_json(SHOTS_OUT, shots_data, include_access=False)

print(f"\nРазмер test.json:  {TEST_OUT.stat().st_size:>10} байт")
print(f"Размер shots.json: {SHOTS_OUT.stat().st_size:>10} байт")



[INFO]  Записан файл data/test.json (258 примеров).
[INFO]  Записан файл data/shots.json (8 примеров).

Размер test.json:      268603 байт
Размер shots.json:       8346 байт


## 10. (Опционально) — посмотреть готовый файл

Читаем `test.json` обратно и показываем первый пример — чтобы убедиться, что всё сериализовалось корректно (кириллица, типы полей, структура).

In [42]:
with open(TEST_OUT, encoding="utf-8") as f:
    loaded = json.load(f)

print(f"access: {loaded['access']}")
print(f"data size: {len(loaded['data'])}")
print(f"\nПервый пример из сохранённого test.json:")
print(json.dumps(loaded["data"][0], indent=2, ensure_ascii=False))

access: public
data size: 258

Первый пример из сохранённого test.json:
{
  "instruction": 0,
  "inputs": {
    "question": "Пусть \\(n\\) — натуральное число, а \\(p\\) — простое. Через \\(n\\) и \\(p\\) обозначим \\(k(n,p)\\) как наибольшее неотрицательное целое число \\(k\\), для которого существует многочлен \\(P(x)\\) с целыми коэффициентами, удовлетворяющий условиям:  \n- Коэффициент при \\(x^n\\) в \\(P(x)\\) равен \\(1\\).  \n- \\(p^k\\) делит \\(P(x)\\) для всех целых \\(x\\).\n\nВычислите\n\\[\n\\sum_{n=11}^{15} \\sum_{p \\in \\{11,13\\}} k(n,p).\n\\]\nкак целое число."
  },
  "outputs": "138",
  "meta": {
    "id": 1,
    "source": "apex",
    "spec": ""
  }
}


## Готово ✅

Что сделано:
- Датасет скачан из приватного `a141592/Luzitania`.
- **Исходный датасет залит как есть** в `MERA-evaluation/Luzitania` (репозиторий создан, если не существовал).
- Структура проверена, примеры показаны.
- Сплиты `test` и `shots` сконвертированы в формат MERA.
- Пройдены sanity-проверки.
- Локальные JSON-файлы сохранены в `./data/test.json` и `./data/shots.json`.

